# Cosmological reference numbers — AGEL J020613$-$011417

Quick reference for the AGEL0206 strong-lens system: light-travel (lookback) times,
ages, distances, angular scales, and lensing geometry for the **deflector** and the
**lensed source**.

**Cosmology:** Planck 2015 flat $\Lambda$CDM, $H_0 = 67.7$, $\Omega_{m,0}=0.302$ — the
project standard adopted to match the companion lens-modeling paper (Ferrami et al.); see
`CLAUDE.md` / `results/PAPER_VALUES.json`.

**Redshifts:** deflector $z_l = 0.67564$ (line-fit systemic, notebook 04); source
$z_s = 1.30263$ ([O II] $\lambda\lambda3726,3729$; the registry rounds this to 1.302 — negligible).

In [1]:
import numpy as np
from astropy.cosmology import FlatLambdaCDM, z_at_value
import astropy.units as u
import astropy.constants as const

# Project-standard cosmology (Planck 2015) — must match PAPER_VALUES.json / the lens paper
cosmo = FlatLambdaCDM(H0=67.7, Om0=0.302)

z_l = 0.67564    # deflector (notebook 04 line-fit systemic)
z_s = 1.30263    # lensed source ([O II] doublet; registry rounds to 1.302)

print(cosmo)
print(f"z_l = {z_l},  z_s = {z_s}")
print(f"Hubble time 1/H0 = {(1/cosmo.H0).to(u.Gyr):.4f}")
print(f"Age of universe today = {cosmo.age(0):.4f}")

FlatLambdaCDM(H0=67.7 km / (Mpc s), Om0=0.302, Tcmb0=0.0 K, Neff=3.04, m_nu=None, Ob0=0.0)
z_l = 0.67564,  z_s = 1.30263
Hubble time 1/H0 = 14.4430 Gyr
Age of universe today = 13.8986 Gyr


## 1. Light-travel (lookback) times

The **lookback time** is how long the light we see today has been travelling — i.e. the
light-travel time from each galaxy to us. The difference $t_{\rm lb}(z_s)-t_{\rm lb}(z_l)$
is how much *earlier* the source light was emitted than the deflector light.

In [2]:
tlb_l = cosmo.lookback_time(z_l)
tlb_s = cosmo.lookback_time(z_s)
age_l = cosmo.age(z_l)
age_s = cosmo.age(z_s)

print(f"Lookback time to deflector (z_l={z_l}):  {tlb_l:.4f}   ({tlb_l.to(u.Gyr).value:.3f} Gyr)")
print(f"Lookback time to source   (z_s={z_s}):  {tlb_s:.4f}   ({tlb_s.to(u.Gyr).value:.3f} Gyr)")
print(f"Source light emitted earlier than deflector by: {(tlb_s-tlb_l):.4f}")
print()
print(f"Age of universe at deflector epoch: {age_l:.4f}")
print(f"Age of universe at source epoch:    {age_s:.4f}")

Lookback time to deflector (z_l=0.67564):  6.3682 Gyr   (6.368 Gyr)
Lookback time to source   (z_s=1.30263):  9.0302 Gyr   (9.030 Gyr)
Source light emitted earlier than deflector by: 2.6620 Gyr

Age of universe at deflector epoch: 7.5304 Gyr
Age of universe at source epoch:    4.8684 Gyr


## 2. Distances

Comoving, angular-diameter ($D_A$), and luminosity ($D_L$) distances, plus the distance
modulus $\mu = 5\log_{10}(D_L/10\,{\rm pc})$.

In [3]:
for name, z in [("deflector", z_l), ("source", z_s)]:
    dc = cosmo.comoving_distance(z)
    da = cosmo.angular_diameter_distance(z)
    dl = cosmo.luminosity_distance(z)
    dm = cosmo.distmod(z)
    print(f"--- {name} (z={z}) ---")
    print(f"  comoving distance D_C      = {dc:.2f}")
    print(f"  angular-diameter dist D_A  = {da:.2f}")
    print(f"  luminosity distance  D_L   = {dl:.2f}")
    print(f"  distance modulus     mu    = {dm:.4f}")
    print()

--- deflector (z=0.67564) ---
  comoving distance D_C      = 2514.90 Mpc
  angular-diameter dist D_A  = 1500.86 Mpc
  luminosity distance  D_L   = 4214.06 Mpc
  distance modulus     mu    = 43.1235 mag

--- source (z=1.30263) ---
  comoving distance D_C      = 4109.42 Mpc
  angular-diameter dist D_A  = 1784.66 Mpc
  luminosity distance  D_L   = 9462.46 Mpc
  distance modulus     mu    = 44.8800 mag



## 3. Angular scale (physical size per arcsec)

Proper transverse kpc per arcsecond at each redshift. (At $z_l$ this is the **7.2764
kpc/arcsec** used throughout the project for $R_e$ etc.)

In [4]:
for name, z in [("deflector", z_l), ("source", z_s)]:
    scale = cosmo.kpc_proper_per_arcmin(z).to(u.kpc/u.arcsec)
    print(f"{name:10s} (z={z}):  {scale:.4f}   (1\" = {scale.value:.4f} kpc)")

deflector  (z=0.67564):  7.2764 kpc / arcsec   (1" = 7.2764 kpc)
source     (z=1.30263):  8.6523 kpc / arcsec   (1" = 8.6523 kpc)


## 4. Lensing geometry

For a lens at $z_l$ and source at $z_s$ we need the **lens$\to$source** angular-diameter
distance $D_{ls}$ (note $D_{ls}\neq D_s-D_l$ in curved spacetime; use `angular_diameter_distance_z1z2`).

- **Lensing distance ratio** $\beta = D_{ls}/D_s$ — the geometric lensing efficiency.
- **Critical surface density** $\Sigma_{\rm cr} = \dfrac{c^2}{4\pi G}\dfrac{D_s}{D_l D_{ls}}$ —
  the convergence-1 mass surface density (reference for the lens mass).

In [5]:
D_l  = cosmo.angular_diameter_distance(z_l)
D_s  = cosmo.angular_diameter_distance(z_s)
D_ls = cosmo.angular_diameter_distance_z1z2(z_l, z_s)

beta = (D_ls / D_s).decompose()
Sigma_cr = (const.c**2 / (4*np.pi*const.G) * D_s/(D_l*D_ls)).to(u.Msun/u.pc**2)

print(f"D_l   (0  -> z_l) = {D_l:.2f}")
print(f"D_s   (0  -> z_s) = {D_s:.2f}")
print(f"D_ls  (z_l-> z_s) = {D_ls:.2f}")
print(f"beta = D_ls/D_s   = {beta:.4f}")
print(f"Sigma_crit        = {Sigma_cr:.2f}")

D_l   (0  -> z_l) = 1500.86 Mpc
D_s   (0  -> z_s) = 1784.66 Mpc
D_ls  (z_l-> z_s) = 692.48 Mpc
beta = D_ls/D_s   = 0.3880
Sigma_crit        = 2855.50 solMass / pc2


## 5. Summary table

In [6]:
from astropy.table import Table
rows = []
for name, z in [("deflector", z_l), ("source", z_s)]:
    rows.append(dict(
        galaxy=name, z=z,
        t_lookback_Gyr=round(cosmo.lookback_time(z).to(u.Gyr).value, 3),
        age_at_z_Gyr=round(cosmo.age(z).to(u.Gyr).value, 3),
        D_C_Mpc=round(cosmo.comoving_distance(z).to(u.Mpc).value, 1),
        D_A_Mpc=round(cosmo.angular_diameter_distance(z).to(u.Mpc).value, 1),
        D_L_Mpc=round(cosmo.luminosity_distance(z).to(u.Mpc).value, 1),
        kpc_per_arcsec=round(cosmo.kpc_proper_per_arcmin(z).to(u.kpc/u.arcsec).value, 4),
        distmod_mag=round(cosmo.distmod(z).value, 3),
    ))
tab = Table(rows)
tab.write("results/cosmology_reference_AGEL0206.csv", overwrite=True)
print("Lensing:  D_ls = %.1f Mpc,  beta = D_ls/D_s = %.4f,  Sigma_crit = %.1f Msun/pc^2"
      % (D_ls.to(u.Mpc).value, beta.value, Sigma_cr.value))
print("Saved -> results/cosmology_reference_AGEL0206.csv")
tab

Lensing:  D_ls = 692.5 Mpc,  beta = D_ls/D_s = 0.3880,  Sigma_crit = 2855.5 Msun/pc^2
Saved -> results/cosmology_reference_AGEL0206.csv


galaxy,z,t_lookback_Gyr,age_at_z_Gyr,D_C_Mpc,D_A_Mpc,D_L_Mpc,kpc_per_arcsec,distmod_mag
str9,float64,float64,float64,float64,float64,float64,float64,float64
deflector,0.67564,6.368,7.53,2514.9,1500.9,4214.1,7.2764,43.124
source,1.30263,9.03,4.868,4109.4,1784.7,9462.5,8.6523,44.88
